In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import snapatac2 as snap
from scipy.sparse import csr_matrix

In [ ]:
he_id = 'CTRL04_2010S1121_75_HE'

In [ ]:
atac = sc.read_h5ad(f'../{he_id}_snapATAC_prediction_fullpeaks_allcells_annotated.h5ad')

In [ ]:
fragments_array = atac.X.copy()
fragments_array_final = fragments_array
fragments_array_final_insertion = np.maximum(0, np.ceil(fragments_array_final-0.05)).astype(int)

In [ ]:
atac.X = csr_matrix(fragments_array_final_insertion.astype(int).copy())

In [ ]:
atac_pbmc = snap.read(snap.datasets.pbmc5k(type='h5ad'), backed=None)
atac_pbmc

In [ ]:
atac.uns['reference_sequences'] = atac_pbmc.uns['reference_sequences'].copy()

In [ ]:
atac.var_names

In [ ]:
import pyranges as pr

gtf = pr.read_gtf("../ATAC/gencode.v38.annotation.gtf")

In [ ]:
genes = gtf[gtf.Feature == "gene"]

In [ ]:
genes

In [ ]:
genes.columns

In [ ]:
genes_start = np.array(genes.Start)
genes_end = np.array(genes.End)
genes_strand = np.array(genes.Strand)
TSS = [x if z == '+' else y for (x, y, z) in zip(genes_start, genes_end, genes_strand)]
genes.TSS = TSS

In [ ]:
### build promoters
promoters = genes.copy()
promoters.Start = genes.TSS - 2000
promoters.End   = genes.TSS + 200
promoters.Name  = promoters.gene_name

In [ ]:
promoters

In [ ]:
peak_df = atac.var_names.to_series().str.split("[:-]", expand=True)
peak_df.columns = ["Chromosome", "Start", "End"]
peak_df["Start"] = peak_df["Start"].astype(int)
peak_df["End"]   = peak_df["End"].astype(int)
peak_df['Index'] = peak_df.index

peaks = pr.PyRanges(peak_df)

In [ ]:
group_signal_norm = group_signal.copy()
for g in group_signal_norm:
    s = group_signal_norm[g]
    group_signal_norm[g] = s / s.max() if s.max() > 0 else s

In [ ]:
np.unique(atac.obs['celltype_final'])

In [ ]:
def gaussian(x, mu, sig, height):
    return height * np.exp(-np.power(x - mu, 2.) / (2 * np.power(sig, 2.)))

In [ ]:
gene_name = "PROX1"#"WFS1"#"GRIA2"#"GRIN2A"#"PRKCD"#"PDGFRB"#"ACTA2"#"CLDN5"#"SOX2"#"MBP"#"AQP4"#"GFAP"#"GAD1"# 
region = genes[genes.gene_name == gene_name]

peaks_sub = peaks.overlap(region)
peak_idx = peaks_sub.df.Index.values 
peak_mat = atac[:, peak_idx].X.toarray() #/ cell_counts[:, 0][:, None]

groups = atac.obs["celltype_final"]
group_names = groups.unique()

group_signal = {}

for g in group_names:
    idx = np.where(groups == g)[0]
    group_signal[g] = peak_mat[idx, :].mean(axis=0)

fig, axs = plt.subplots(10, 1, figsize=(8, 20))

for g in range(10):
    lower_y = -0.002
    upper_y = 0.4
    
    # Plot each group's pseudo coverage
    xs = peaks_sub.df.Start.values
    xe = peaks_sub.df.End.values
    # 1. Get the signal for the current group
    signal = group_signal[group_names[g]]
    
    # 2. Cap the signal at upper_y (0.4)
    # This keeps values between 0 and 0.4
    ys = np.clip(signal, a_min=None, a_max=upper_y)

    '''
    # plot rectangles
    for x1, x2, h in zip(xs, xe, ys):
        axs[g].fill_between([x1, x2], 0, h, alpha=0.5, label=g)
    '''
    # plot gaussian curves
    for x1, x2, h in zip(xs, xe, ys):
        if h > 0: # Only plot if there is signal
            mu = (x1 + x2) / 2
            # Standard deviation: adjust the divisor to make curves wider or thinner
            # (x2 - x1) / 4 ensures most of the curve fits within the peak boundaries
            sig = (x2 - x1) / 4 
            
            # Create a fine x-axis for a smooth curve
            x_range = np.linspace(x1 - (sig*2), x2 + (sig*2), 100)
            y_range = gaussian(x_range, mu, sig, h)
            
            axs[g].fill_between(x_range, 0, y_range, alpha=0.5, color='steelblue')
                
    # Gene bodies overlapping region
    genes_sub = region.df#genes.overlap(region).df
    
    for _, row in genes_sub.iterrows():
        axs[g].hlines(
            y=upper_y/40,
            xmin=row.Start,
            xmax=row.End,
            color='black',
            linewidth=3
        )
        axs[g].text(
            (row.Start + row.End)/2,
            lower_y/4+0.3, #0.1,#-0.15,
            row.gene_name+' in '+group_names[g],
            ha="center",
            fontsize=20
        )
    
    # Promoters
    #prom_sub = promoters.overlap(region).df
    prom_sub = promoters[promoters.Name==gene_name].df
    
    for _, row in prom_sub.iterrows():
        axs[g].add_patch(plt.Rectangle(
            (row.Start, upper_y/40),#-0.05
            row.End - row.Start,
            upper_y/40,
            facecolor="firebrick",
            alpha=0.7
        ))
    
    axs[g].set_ylim(lower_y, upper_y)
    axs[g].set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}")
    axs[g].set_ylabel("Accessibility")
    #axs[g].set_title(f"Pseudo-CoveragePlot of gene {gene_name} in cluster {group_names[g]}")
    # Remove top and right spines
    axs[g].spines['top'].set_visible(False)
    axs[g].spines['right'].set_visible(False)
    
plt.tight_layout()
plt.show()

In [ ]:
gene_name = "PROX1"#"WFS1"#"GRIA2"#"GRIN2A"#"PRKCD"#"PDGFRB"#"ACTA2"#"CLDN5"#"SOX2"#"MBP"#"AQP4"#"GFAP"#"GAD1"# 
region = genes[genes.gene_name == gene_name]

peaks_sub = peaks.overlap(region)
peak_idx = peaks_sub.df.Index.values 
peak_mat = atac[:, peak_idx].X.toarray() #/ cell_counts[:, 0][:, None]

groups = atac.obs["celltype_final"]
group_names = ['GC_Neuron']#groups.unique()

group_signal = {}

for g in group_names:
    idx = np.where(groups == g)[0]
    group_signal[g] = peak_mat[idx, :].mean(axis=0)

fig, axs = plt.subplots(1, 1, figsize=(8, 2.5))

#for g in range(2):
g = 0
lower_y = -0.002
upper_y = 0.4

# Plot each group's pseudo coverage
xs = peaks_sub.df.Start.values
xe = peaks_sub.df.End.values
# 1. Get the signal for the current group
signal = group_signal[group_names[g]]

# 2. Cap the signal at upper_y (0.4)
# This keeps values between 0 and 0.4
ys = np.clip(signal, a_min=None, a_max=upper_y)

'''
# plot rectangles
for x1, x2, h in zip(xs, xe, ys):
    axs[g].fill_between([x1, x2], 0, h, alpha=0.5, label=g)
'''
# plot gaussian curves
for x1, x2, h in zip(xs, xe, ys):
    if h > 0: # Only plot if there is signal
        mu = (x1 + x2) / 2
        # Standard deviation: adjust the divisor to make curves wider or thinner
        # (x2 - x1) / 4 ensures most of the curve fits within the peak boundaries
        sig = (x2 - x1) / 8 
        
        # Create a fine x-axis for a smooth curve
        x_range = np.linspace(x1 - (sig*2), x2 + (sig*2), 100)
        y_range = gaussian(x_range, mu, sig, h)
        
        axs.fill_between(x_range, 0, y_range, alpha=0.5, color='steelblue')
            
# Gene bodies overlapping region
genes_sub = region.df#genes.overlap(region).df

for _, row in genes_sub.iterrows():
    axs.hlines(
        y=upper_y/40,
        xmin=row.Start,
        xmax=row.End,
        color='black',
        linewidth=3
    )
    axs.text(
        (row.Start + row.End)/2,
        lower_y/4+0.3, #0.1,#-0.15,
        row.gene_name+' in '+group_names[g],
        ha="center",
        fontsize=20
    )

# Promoters
#prom_sub = promoters.overlap(region).df
prom_sub = promoters[promoters.Name==gene_name].df

for _, row in prom_sub.iterrows():
    axs.add_patch(plt.Rectangle(
        (row.Start, upper_y/40),#-0.05
        row.End - row.Start,
        upper_y/40,
        facecolor="firebrick",
        alpha=0.7
    ))

axs.set_ylim(lower_y, upper_y)
axs.set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)
axs.set_ylabel("Accessibility", fontsize=18)
#axs[g].set_title(f"Pseudo-CoveragePlot of gene {gene_name} in cluster {group_names[g]}")
# Remove top and right spines
axs.spines['top'].set_visible(False)
axs.spines['right'].set_visible(False)

#axs[1].set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)    
plt.tight_layout()
plt.show()

In [ ]:
gene_name = "WFS1"#"PROX1"#"GRIA2"#"GRIN2A"#"PRKCD"#"PDGFRB"#"ACTA2"#"CLDN5"#"SOX2"#"MBP"#"AQP4"#"GFAP"#"GAD1"# 
region = genes[genes.gene_name == gene_name]

peaks_sub = peaks.overlap(region)
peak_idx = peaks_sub.df.Index.values 
peak_mat = atac[:, peak_idx].X.toarray() #/ cell_counts[:, 0][:, None]

groups = atac.obs["celltype_final"]
group_names = ['CA_Neuron', 'Neuron_Other']#groups.unique()

group_signal = {}

for g in group_names:
    idx = np.where(groups == g)[0]
    group_signal[g] = peak_mat[idx, :].mean(axis=0)

fig, axs = plt.subplots(2, 1, figsize=(8, 5))

for g in range(2):
    lower_y = -0.002
    upper_y = 0.8
    
    # Plot each group's pseudo coverage
    xs = peaks_sub.df.Start.values
    xe = peaks_sub.df.End.values
    # 1. Get the signal for the current group
    signal = group_signal[group_names[g]]
    
    # 2. Cap the signal at upper_y (0.4)
    # This keeps values between 0 and 0.4
    ys = np.clip(signal, a_min=None, a_max=upper_y)

    '''
    # plot rectangles
    for x1, x2, h in zip(xs, xe, ys):
        axs[g].fill_between([x1, x2], 0, h, alpha=0.5, label=g)
    '''
    # plot gaussian curves
    for x1, x2, h in zip(xs, xe, ys):
        if h > 0: # Only plot if there is signal
            mu = (x1 + x2) / 2
            # Standard deviation: adjust the divisor to make curves wider or thinner
            # (x2 - x1) / 4 ensures most of the curve fits within the peak boundaries
            sig = (x2 - x1) / 8 
            
            # Create a fine x-axis for a smooth curve
            x_range = np.linspace(x1 - (sig*2), x2 + (sig*2), 100)
            y_range = gaussian(x_range, mu, sig, h)
            
            axs[g].fill_between(x_range, 0, y_range, alpha=0.5, color='steelblue')
                
    # Gene bodies overlapping region
    genes_sub = region.df#genes.overlap(region).df
    
    for _, row in genes_sub.iterrows():
        axs[g].hlines(
            y=upper_y/40,
            xmin=row.Start,
            xmax=row.End,
            color='black',
            linewidth=3
        )
        axs[g].text(
            (row.Start + row.End)/2,
            lower_y/4+0.7, #0.1,#-0.15,
            row.gene_name+' in '+group_names[g],
            ha="center",
            fontsize=20
        )
    
    # Promoters
    #prom_sub = promoters.overlap(region).df
    prom_sub = promoters[promoters.Name==gene_name].df
    
    for _, row in prom_sub.iterrows():
        axs[g].add_patch(plt.Rectangle(
            (row.Start, upper_y/40),#-0.05
            row.End - row.Start,
            upper_y/40,
            facecolor="firebrick",
            alpha=0.7
        ))
    
    axs[g].set_ylim(lower_y, upper_y)
    #axs[g].set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)
    axs[g].set_ylabel("Accessibility", fontsize=18)
    #axs[g].set_title(f"Pseudo-CoveragePlot of gene {gene_name} in cluster {group_names[g]}")
    # Remove top and right spines
    axs[g].spines['top'].set_visible(False)
    axs[g].spines['right'].set_visible(False)

axs[1].set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)    
plt.tight_layout()
plt.show()

In [ ]:
gene_name = "CNP"#"WFS1"#"GRIA2"#"GRIN2A"#"PRKCD"#"PDGFRB"#"ACTA2"#"CLDN5"#"SOX2"#"MBP"#"AQP4"#"GFAP"#"GAD1"# 
region = genes[genes.gene_name == gene_name]

peaks_sub = peaks.overlap(region)
peak_idx = peaks_sub.df.Index.values 
peak_mat = atac[:, peak_idx].X.toarray() #/ cell_counts[:, 0][:, None]

groups = atac.obs["celltype_final"]
group_names = ['Oligo']#groups.unique()

group_signal = {}

for g in group_names:
    idx = np.where(groups == g)[0]
    group_signal[g] = peak_mat[idx, :].mean(axis=0)

fig, axs = plt.subplots(1, 1, figsize=(8, 2.5))

#for g in range(2):
g = 0
lower_y = -0.002
upper_y = 0.52

# Plot each group's pseudo coverage
xs = peaks_sub.df.Start.values
xe = peaks_sub.df.End.values
# 1. Get the signal for the current group
signal = group_signal[group_names[g]]

# 2. Cap the signal at upper_y (0.4)
# This keeps values between 0 and 0.4
ys = np.clip(signal, a_min=None, a_max=upper_y)

'''
# plot rectangles
for x1, x2, h in zip(xs, xe, ys):
    axs[g].fill_between([x1, x2], 0, h, alpha=0.5, label=g)
'''
# plot gaussian curves
for x1, x2, h in zip(xs, xe, ys):
    if h > 0: # Only plot if there is signal
        mu = (x1 + x2) / 2
        # Standard deviation: adjust the divisor to make curves wider or thinner
        # (x2 - x1) / 4 ensures most of the curve fits within the peak boundaries
        sig = (x2 - x1) / 8 
        
        # Create a fine x-axis for a smooth curve
        x_range = np.linspace(x1 - (sig*2), x2 + (sig*2), 100)
        y_range = gaussian(x_range, mu, sig, h)
        
        axs.fill_between(x_range, 0, y_range, alpha=0.5, color='steelblue')
            
# Gene bodies overlapping region
genes_sub = region.df#genes.overlap(region).df

for _, row in genes_sub.iterrows():
    axs.hlines(
        y=upper_y/40,
        xmin=row.Start,
        xmax=row.End,
        color='black',
        linewidth=3
    )
    axs.text(
        (row.Start + row.End)/2,
        lower_y/4+0.4, #0.1,#-0.15,
        row.gene_name+' in '+group_names[g],
        ha="center",
        fontsize=20
    )

# Promoters
#prom_sub = promoters.overlap(region).df
prom_sub = promoters[promoters.Name==gene_name].df

for _, row in prom_sub.iterrows():
    axs.add_patch(plt.Rectangle(
        (row.Start, upper_y/40),#-0.05
        row.End - row.Start,
        upper_y/40,
        facecolor="firebrick",
        alpha=0.7
    ))

axs.set_ylim(lower_y, upper_y)
axs.set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)
axs.set_ylabel("Accessibility", fontsize=18)
#axs[g].set_title(f"Pseudo-CoveragePlot of gene {gene_name} in cluster {group_names[g]}")
# Remove top and right spines
axs.spines['top'].set_visible(False)
axs.spines['right'].set_visible(False)

#axs[1].set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)    
plt.tight_layout()
plt.show()

In [ ]:
gene_name = "PDGFRB"#"WFS1"#"GRIA2"#"GRIN2A"#"PRKCD"#"PDGFRB"#"ACTA2"#"CLDN5"#"SOX2"#"MBP"#"AQP4"#"GFAP"#"GAD1"# 
region = genes[genes.gene_name == gene_name]

peaks_sub = peaks.overlap(region)
peak_idx = peaks_sub.df.Index.values 
peak_mat = atac[:, peak_idx].X.toarray() #/ cell_counts[:, 0][:, None]

groups = atac.obs["celltype_final"]
group_names = ['Vascular']#groups.unique()

group_signal = {}

for g in group_names:
    idx = np.where(groups == g)[0]
    group_signal[g] = peak_mat[idx, :].mean(axis=0)

fig, axs = plt.subplots(1, 1, figsize=(8, 2.5))

#for g in range(2):
g = 0
lower_y = -0.002
upper_y = 0.15

# Plot each group's pseudo coverage
xs = peaks_sub.df.Start.values
xe = peaks_sub.df.End.values
# 1. Get the signal for the current group
signal = group_signal[group_names[g]]

# 2. Cap the signal at upper_y (0.4)
# This keeps values between 0 and 0.4
ys = np.clip(signal, a_min=None, a_max=upper_y)

'''
# plot rectangles
for x1, x2, h in zip(xs, xe, ys):
    axs[g].fill_between([x1, x2], 0, h, alpha=0.5, label=g)
'''
# plot gaussian curves
for x1, x2, h in zip(xs, xe, ys):
    if h > 0: # Only plot if there is signal
        mu = (x1 + x2) / 2
        # Standard deviation: adjust the divisor to make curves wider or thinner
        # (x2 - x1) / 4 ensures most of the curve fits within the peak boundaries
        sig = (x2 - x1) / 8 
        
        # Create a fine x-axis for a smooth curve
        x_range = np.linspace(x1 - (sig*2), x2 + (sig*2), 100)
        y_range = gaussian(x_range, mu, sig, h)
        
        axs.fill_between(x_range, 0, y_range, alpha=0.5, color='steelblue')
            
# Gene bodies overlapping region
genes_sub = region.df#genes.overlap(region).df

for _, row in genes_sub.iterrows():
    axs.hlines(
        y=upper_y/40,
        xmin=row.Start,
        xmax=row.End,
        color='black',
        linewidth=3
    )
    axs.text(
        (row.Start + row.End)/2,
        lower_y/4+0.12, #0.1,#-0.15,
        row.gene_name+' in '+group_names[g],
        ha="center",
        fontsize=20
    )

# Promoters
#prom_sub = promoters.overlap(region).df
prom_sub = promoters[promoters.Name==gene_name].df

for _, row in prom_sub.iterrows():
    axs.add_patch(plt.Rectangle(
        (row.Start, upper_y/40),#-0.05
        row.End - row.Start,
        upper_y/40,
        facecolor="firebrick",
        alpha=0.7
    ))

axs.set_ylim(lower_y, upper_y)
axs.set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)
axs.set_ylabel("Accessibility", fontsize=18)
#axs[g].set_title(f"Pseudo-CoveragePlot of gene {gene_name} in cluster {group_names[g]}")
# Remove top and right spines
axs.spines['top'].set_visible(False)
axs.spines['right'].set_visible(False)

#axs[1].set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)    
plt.tight_layout()
plt.show()

In [ ]:
gene_name = "AQP1"#"WFS1"#"GRIA2"#"GRIN2A"#"PRKCD"#"PDGFRB"#"ACTA2"#"CLDN5"#"SOX2"#"MBP"#"AQP4"#"GFAP"#"GAD1"# 
region = genes[genes.gene_name == gene_name]

peaks_sub = peaks.overlap(region)
peak_idx = peaks_sub.df.Index.values 
peak_mat = atac[:, peak_idx].X.toarray() #/ cell_counts[:, 0][:, None]

groups = atac.obs["celltype_final"]
group_names = ['Choroid']#groups.unique()

group_signal = {}

for g in group_names:
    idx = np.where(groups == g)[0]
    group_signal[g] = peak_mat[idx, :].mean(axis=0)

fig, axs = plt.subplots(1, 1, figsize=(8, 2.5))

#for g in range(2):
g = 0
lower_y = -0.002
upper_y = 0.5

# Plot each group's pseudo coverage
xs = peaks_sub.df.Start.values
xe = peaks_sub.df.End.values
# 1. Get the signal for the current group
signal = group_signal[group_names[g]]

# 2. Cap the signal at upper_y (0.4)
# This keeps values between 0 and 0.4
ys = np.clip(signal, a_min=None, a_max=upper_y)

'''
# plot rectangles
for x1, x2, h in zip(xs, xe, ys):
    axs[g].fill_between([x1, x2], 0, h, alpha=0.5, label=g)
'''
# plot gaussian curves
for x1, x2, h in zip(xs, xe, ys):
    if h > 0: # Only plot if there is signal
        mu = (x1 + x2) / 2
        # Standard deviation: adjust the divisor to make curves wider or thinner
        # (x2 - x1) / 4 ensures most of the curve fits within the peak boundaries
        sig = (x2 - x1) / 8 
        
        # Create a fine x-axis for a smooth curve
        x_range = np.linspace(x1 - (sig*2), x2 + (sig*2), 100)
        y_range = gaussian(x_range, mu, sig, h)
        
        axs.fill_between(x_range, 0, y_range, alpha=0.5, color='steelblue')
            
# Gene bodies overlapping region
genes_sub = region.df#genes.overlap(region).df

for _, row in genes_sub.iterrows():
    axs.hlines(
        y=upper_y/40,
        xmin=row.Start,
        xmax=row.End,
        color='black',
        linewidth=3
    )
    axs.text(
        (row.Start + row.End)/2,
        lower_y/4+0.5, #0.1,#-0.15,
        row.gene_name+' in '+group_names[g],
        ha="center",
        fontsize=20
    )

# Promoters
#prom_sub = promoters.overlap(region).df
prom_sub = promoters[promoters.Name==gene_name].df

for _, row in prom_sub.iterrows():
    axs.add_patch(plt.Rectangle(
        (row.Start, upper_y/40),#-0.05
        row.End - row.Start,
        upper_y/40,
        facecolor="firebrick",
        alpha=0.7
    ))

axs.set_ylim(lower_y, upper_y)
axs.set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)
axs.set_ylabel("Accessibility", fontsize=18)
#axs[g].set_title(f"Pseudo-CoveragePlot of gene {gene_name} in cluster {group_names[g]}")
# Remove top and right spines
axs.spines['top'].set_visible(False)
axs.spines['right'].set_visible(False)

#axs[1].set_xlabel(f"Genomic position: {np.array(genes_sub)[0][0]}: {np.array(genes_sub)[0][3]} - {np.array(genes_sub)[0][4]}", fontsize=20)    
plt.tight_layout()
plt.show()

### motif plots - excluding choroid plexus

In [ ]:
cell_x = np.array(atac.obs['x'].copy())
cell_y = np.array(atac.obs['y'].copy())

choroid_x = np.array(atac.obs['x'][celltype_final=='Choroid'])
choroid_y = np.array(atac.obs['y'][celltype_final=='Choroid'])

dist_to_choroid = []
for i in range(len(cell_x)):
    tmp_min_dist = np.min(np.sqrt((cell_x[i]-choroid_x)**2 + (cell_y[i]-choroid_y)**2))
    dist_to_choroid.append(tmp_min_dist)
    
include_cells = np.array([True] * atac.n_obs)
include_cells[celltype_final=='Choroid'] = False
include_cells[(celltype_final=='LM_glia')] = False
include_cells[(celltype_final=='Unknown')] = False
include_cells[(celltype_final=='Vascular')*(atac.obs['y']>= (-5000./15000 * (atac.obs['x']- 15000) + 25000))] = False #
include_cells[(np.array(dist_to_choroid)<500)*(atac.obs['y']>= (-11250./15000 * (atac.obs['x']- 20000) + 30000))] = False
atac.obs['include_cells'] = include_cells

In [ ]:
df_pychromvar= pd.read_csv('{he_id}_df_pychromvar.csv', index_col=0)
adata_motifs_full = ad.AnnData(X = df_pychromvar)
adata_motifs_full.obs['celltype'] = atac.obs['celltype_final'].copy()
adata_motifs = adata_motifs_full[include_cells, :].copy()
adata_motifs

In [ ]:
from PIL import Image, ImageOps
import matplotlib.pyplot as plt 
brain_image = Image.open(f'../CellViT/Brain/preprocessing/2010S1121_75_HE/2010S1121_75_HE/thumbnails/thumbnail_mpp_5.png')
pil_he_img = Image.fromarray(np.array(brain_image))
invert_he_img = ImageOps.invert(pil_he_img)
gray_he_img = invert_he_img.convert('LA')
gray_he_img_array = np.array(gray_he_img)
gray_he_img_final = np.stack((gray_he_img_array[:, :, 0],)*3, axis=-1)
plt.imshow(gray_he_img_final)
plt.axis('off')

In [ ]:
gene_cluster.uns['spatial']['CTRL04_2010S1121_75_HE']['images']['hires']=np.array(brain_image)
gene_cluster.uns['spatial']['CTRL04_2010S1121_75_HE']['images']['invert']=gray_he_img_final
gene_cluster.uns['spatial']['CTRL04_2010S1121_75_HE']['scalefactors']['tissue_hires_scalef'] = 1652/33057
gene_cluster.uns['spatial']['CTRL04_2010S1121_75_HE']["scalefactors"]['tissue_invert_scalef'] = gene_cluster.uns['spatial']['CTRL04_2010S1121_75_HE']["scalefactors"]['tissue_hires_scalef']

In [ ]:
adata_motifs_full.uns['spatial'] = gene_cluster.uns['spatial'].copy()

In [ ]:
import json

he_dir = '../CellViT/Brain/preprocessing/2010S1121_75_HE/2010S1121_75_HE/cell_detection'
he_id = 'CTRL04_2010S1121_75_HE'

with open(f'{he_dir}/cells.json') as f:
    cell_data = json.load(f)

## extract one patch from cell_data
cell_centroid_0 = []
cell_centroid_1 = []
patch_coords = []
offset1 = []
offset2 = []
for cell in cell_data['cells']:
    cell_centroid_0.append(cell['centroid'][0])
    cell_centroid_1.append(cell['centroid'][1])
    patch_coords.append(str(cell['patch_coordinates'][0])+'_'+str(cell['patch_coordinates'][1]))
    offset1.append(cell['offset_global'][0])
    offset2.append(cell['offset_global'][1])

In [ ]:
adata_motifs_full.obsm['spatial'] = np.array([cell_centroid_0, cell_centroid_1]).T.copy()

In [ ]:
sc.set_figure_params(facecolor='black', frameon=False, dpi=300)
motif = 'MA1128.2.FOSL1::JUN'#"VIP"#'SST'#'CX3CR1'#'TMEM119'#'PECAM1'#'SLC1A2'#'RGS14'#'P2RY12'#'CLDN5'#
fig = sc.pl.spatial(adata_motifs_full[include_cells, :], color=motif, size=4.5, img_key='invert', alpha_img=1.0, legend_loc=None,frameon=False, colorbar_loc=None, title='', show=False, return_fig=True, vmin=1)

In [ ]:
sc.set_figure_params(facecolor='black', frameon=False, dpi=300)
motif = 'MA0670.2.NFIA'#"VIP"#'SST'#'CX3CR1'#'TMEM119'#'PECAM1'#'SLC1A2'#'RGS14'#'P2RY12'#'CLDN5'#
fig = sc.pl.spatial(adata_motifs_full[include_cells, :], color=motif, size=4.5, img_key='invert', alpha_img=1.0, legend_loc=None,frameon=False, colorbar_loc=None, title='', show=False, return_fig=True, vmin=1)

In [ ]:
sc.set_figure_params(facecolor='black', frameon=False, dpi=300)
motif = 'MA0623.2.NEUROG1'#"VIP"#'SST'#'CX3CR1'#'TMEM119'#'PECAM1'#'SLC1A2'#'RGS14'#'P2RY12'#'CLDN5'#
fig = sc.pl.spatial(adata_motifs_full[include_cells, :], color=motif, size=4.5, img_key='invert', alpha_img=1.0, legend_loc=None,frameon=False, colorbar_loc=None, title='', show=False, return_fig=True, vmin=1)

In [ ]:
sc.set_figure_params(facecolor='black', frameon=False, dpi=300)
motif = 'MA0442.3.SOX10'#"VIP"#'SST'#'CX3CR1'#'TMEM119'#'PECAM1'#'SLC1A2'#'RGS14'#'P2RY12'#'CLDN5'#
fig = sc.pl.spatial(adata_motifs_full[include_cells, :], color=motif, size=4.5, img_key='invert', alpha_img=1.0, legend_loc=None,frameon=False, colorbar_loc=None, title='', show=False, return_fig=True, vmin=0)